# 城市科技 Python 第 10 課：空氣品質 AQI 資料專題
臺北市芳和實驗中學 高一第二學期 ｜ 許惠淑老師

**本課目標**
- 會用 requests 呼叫 API、解析 JSON
- 會用 Pandas 篩選測站資料
- 會把程式包成互動網頁（Gradio）並完成專題

> 📌 開始前：請先按「**檔案 → 在雲端硬碟中儲存副本**」，再在自己的副本上作答，否則關掉就不見了！
> 💻 線上練習（Online Judge）：[http://103.226.213.97/course/view.php?id=2109](http://103.226.213.97/course/view.php?id=2109)

## 一、什麼是 API？
API 就像餐廳的點餐窗口：你照規定的格式「點餐」（送出網址），伺服器就把資料（通常是 JSON）送回來。

本課使用 **環境部環境資料開放平臺** 的「空氣品質指標(AQI)」資料集 `aqx_p_432`。
1. 到 [環境資料開放平臺](https://data.moenv.gov.tw/) 註冊會員，取得自己的 **API Key**
2. 在 Colab 左側 🔑「Secrets（密鑰）」新增名稱 `MOENV_API_KEY`，值貼上 API Key，並開啟「筆記本存取權」

> ⚠️ 不要把 API Key、密碼直接寫在程式碼裡再分享出去！

In [ ]:
import requests
import pandas as pd
from google.colab import userdata

API_KEY = userdata.get("MOENV_API_KEY")
url = f"https://data.moenv.gov.tw/api/v2/aqx_p_432?api_key={API_KEY}&limit=1000&format=JSON"
r = requests.get(url, timeout=30)
data = r.json()
records = data["records"] if isinstance(data, dict) else data
print("測站數：", len(records))
records[0]

## ✏️ 實作 1：用迴圈輸出每個測站的 縣市、站名、AQI、狀態

In [ ]:
# ✏️ 練習
# 提示：每一筆 rec 是字典，用 rec["county"]、rec["sitename"]、rec["aqi"]、rec["status"]

## 二、轉成 DataFrame

In [ ]:
df = pd.DataFrame(records)
df["aqi"] = pd.to_numeric(df["aqi"], errors="coerce")   # 轉數字，無法轉的變 NaN
df = df.dropna(subset=["aqi"])
df[["county", "sitename", "aqi", "status", "pm2.5", "publishtime"]].head()

## ✏️ 實作 2：(1) 臺北市所有測站 AQI 由高到低　(2) 全臺 AQI 最高的 5 個測站　(3) 古亭站目前的 AQI

In [ ]:
# ✏️ 練習
# (1)

# (2)

# (3)

## 三、校內空氣盒子（LASS 開源感測資料）
學校的空氣盒子 ID：`B827EB63D1C8`（[網頁版](https://pm25.lass-net.org/data/show.php?device_id=B827EB63D1C8)）。資料時間是 UTC，要 +8 小時才是台灣時間。

若盒子斷電或斷網就會沒有資料，可到 [LASS 地圖](https://pm25.lass-net.org/) 找附近其他盒子的 ID。

In [ ]:
from datetime import datetime, timedelta
DEVICE = "B827EB63D1C8"
url = f"https://pm25.lass-net.org/API-1.0.0/device/{DEVICE}/latest/?format=JSON"
feeds = requests.get(url, timeout=30).json()["feeds"]
box = None
if not feeds:
    print("空氣盒子目前沒有上傳資料，請換一個附近的盒子 ID")
else:
    box = list(feeds[0].values())[0]
    t = datetime.strptime(f'{box["date"]} {box["time"]}', "%Y-%m-%d %H:%M:%S") + timedelta(hours=8)
    print("時間", t)
    print("PM2.5", box.get("s_d0"), "溫度", box.get("s_t0"), "濕度", box.get("s_h0"))

## ✏️ 實作 3：比較校內空氣盒子 PM2.5 和最近的官方測站（古亭）pm2.5，誰比較高？差多少？
（空氣盒子是低成本感測器，數值會有誤差，這也是資料分析要注意的地方。）

In [ ]:
# ✏️ 練習
# 在這裡寫程式

## 四、超過門檻就寄信通知
> 📢 LINE Notify 已於 2025 年 3 月 31 日停止服務，舊範例不能再用。這裡改用 Gmail 寄信。

1. Google 帳戶要開啟兩步驟驗證，再建立「[應用程式密碼](https://support.google.com/accounts/answer/185833)」（學校帳號若無法建立，請用個人帳號）
2. 在 Colab Secrets 新增 `GMAIL_ADDRESS`、`GMAIL_APP_PASSWORD`

In [ ]:
import smtplib
from email.mime.text import MIMEText

def send_email(subject, body, to):
    sender = userdata.get("GMAIL_ADDRESS")
    msg = MIMEText(body, "plain", "utf-8")
    msg["Subject"], msg["From"], msg["To"] = subject, sender, to
    with smtplib.SMTP("smtp.gmail.com", 587) as s:
        s.starttls()
        s.login(sender, userdata.get("GMAIL_APP_PASSWORD"))
        s.send_message(msg)

In [ ]:
# ✏️ 練習
# 實作 4：古亭 AQI 超過 100（對敏感族群不健康）就寄信給自己，否則輸出目前 AQI

## 五、Gradio：把程式變成互動網頁
Gradio 可以把一個函式變成有輸入框、按鈕的網頁，執行後會出現一個暫時的分享網址；有效期限有限，Colab 停止執行後就會失效。

In [ ]:
!pip -q install gradio

In [ ]:
import gradio as gr

def bmi(h, w):
    b = w / (h / 100) ** 2
    level = "過輕" if b < 18.5 else "正常範圍" if b < 24 else "過重" if b < 27 else "肥胖"
    return f"你的 BMI 是 {b:.2f}，{level}"

gr.Interface(bmi,
             inputs=[gr.Number(label="身高（公分）"), gr.Number(label="體重（公斤）")],
             outputs="text",
             title="BMI 小幫手").launch()

## ✏️ 實作 5：空氣品質查詢器——輸入測站名稱，回傳該站 AQI、狀態與一句建議

In [ ]:
# ✏️ 練習
def query(site):
    # 用 df 找出該站資料；找不到要回傳「查無此測站」
    pass

# gr.Interface(query, inputs=gr.Textbox(label="測站名稱"), outputs="text", title="空氣品質查詢器").launch()

## 🎯 期末專題：用 Python 做一個「與人互動」的資料小工具
**主題自選**（AQI、YouBike、天氣、校內空氣盒子、自己的資料…），必須包含：
1. 取得資料：API 或開放資料檔（註明來源與取得日期）
2. 處理資料：至少 1 個篩選或統計
3. 呈現：1 張圖表 **或** 1 個 Gradio 互動介面
4. 文字說明：動機、做法、發現、遇到的困難

繳交：Colab 共用連結（檢視者權限）＋ 1 分鐘展示。